<h1>1.1 Criação da tabela gold.ft_vendas_consumidor_local</h1>

In [0]:
%sql
-- criação do schema gold
USE CATALOG medalhao;
CREATE DATABASE IF NOT EXISTS gold;


# 📈 1º Projeto: Área de Logística - Vendas por Localidade

O objetivo deste projeto é consolidar dados de pedidos e localização dos consumidores na camada **Gold**, criando as estruturas necessárias para que a área de Logística possa identificar as cidades e estados com maior concentração de vendas.

Esta análise é crucial para a **otimização de rotas de entrega** e o **planejamento de centros de distribuição** .

## Entregáveis:
1.  **Tabela Fato:** `gold.ft_vendas_consumidor_local` (Granularidade: Pedido) 
2.  **View Agregada:** `gold.view_total_compras_por_consumidor` (Agregação: Localidade) 
3.  **Consulta de Negócio:** Total de vendas por Estado 

A tabela Fato mantém o histórico detalhado, enquanto a View realiza o consolidado para análise imediat

In [0]:
from pyspark.sql.functions import col, date_format

In [0]:
%sql
-- Criação da tabela Fato gold.ft_vendas_consumidor_local (mantém o histórico de pedidos)
CREATE OR REPLACE TABLE gold.ft_vendas_consumidor_local
USING DELTA
OPTIONS ('mode' 'overwrite')
AS
SELECT
    t1.id_pedido,
    t2.id_consumidor,
    t1.valor_total_pago_brl AS valor_total_pedido_brl, -- Usando nome real da silver e aliásando para o nome final
    t2.cidade,
    t2.estado,
    t1.data AS data_pedido -- Usando nome real da silver e aliásando para o nome final
FROM
    silver.ft_pedido_total AS t1
INNER JOIN
    silver.ft_consumidores AS t2
ON
    t1.id_consumidor = t2.id_consumidor;

In [0]:
%sql
-- Criação da View gold.view_total_compras_por_consumidor para consolidação de vendas por localidade
CREATE OR REPLACE VIEW gold.view_total_compras_por_consumidor
AS
SELECT
    cidade,
    estado,
    COUNT(DISTINCT id_pedido) AS quantidade_vendas,
    SUM(valor_total_pedido_brl) AS valor_total_localidade
FROM
    gold.ft_vendas_consumidor_local
GROUP BY
    cidade,
    estado
ORDER BY
    valor_total_localidade DESC;

<h1>2º Projeto - Área de Logística (Análise de Atrasos de Entregas)</h1>



# 🚚 Resumo dos Objetivos das Views da Área de Logística

---

## 📦 `gold.view_tempo_medio_entrega_localidade`

| Objetivo Principal | Indicadores Fornecidos | Foco Estratégico |
| :--- | :--- | :--- |
| **Avaliar a Eficiência Geográfica.** | `tempo_medio_entrega` (real) | Identificar quais **Cidades** e **Estados** possuem os maiores tempos de entrega (comparado ao estimado). |
| **Comparar Desempenho vs. Expectativa.** | `tempo_medio_estimado` (prometido) | **KPI Chave:** `entrega_maior_que_estimado` (SIM/NÃO). Ajuda a direcionar recursos para locais onde o prazo real consistentemente excede a promessa. |

---

## 🥇 `gold.view_vendedor_pontualidade`

| Objetivo Principal | Indicadores Fornecidos | Foco Estratégico |
| :--- | :--- | :--- |
| **Ranqueamento e Gestão de Performance.** | `total_pedidos`, `total_atrasados` | Criar um ranking para monitorar e gerenciar a **qualidade operacional** dos vendedores. |
| **Calcular o Risco de Atraso.** | **`percentual_atraso`** | Foca no percentual de pedidos que falharam em cumprir o prazo de entrega. Crucial para incentivos ou penalidades a vendedores. |

In [0]:
%sql
-- Criação da tabela Fato gold.ft_atrasos_pedidos_local_vendedor com as colunas de timestamp corrigidas
CREATE OR REPLACE TABLE gold.ft_atrasos_pedidos_local_vendedor
USING DELTA
OPTIONS ('mode' 'overwrite')
AS
SELECT
    t1.id_pedido,
    t3.id_vendedor,
    t2.id_consumidor,
    CASE
        -- Usando o nome exato da coluna da estimativa de entrega
        WHEN t1.pedido_entregue_timestamp > t1.pedido_estimativa_entrega_timestamp THEN 'Não'
        WHEN t1.status = 'entregue' THEN 'Sim'
        ELSE 'Não Entregue'
    END AS entrega_no_prazo,
    -- Usando os nomes exatos para o cálculo do tempo real e estimado
    DATEDIFF(t1.pedido_entregue_timestamp, t1.pedido_aprovado_timestamp) AS tempo_entrega_dias,
    DATEDIFF(t1.pedido_estimativa_entrega_timestamp, t1.pedido_aprovado_timestamp) AS tempo_entrega_estimado_dias,
    t2.cidade,
    t2.estado
FROM
    silver.ft_pedidos AS t1
INNER JOIN
    silver.ft_consumidores AS t2
ON
    t1.id_consumidor = t2.id_consumidor
INNER JOIN (
    SELECT DISTINCT id_pedido, id_vendedor FROM silver.ft_itens_pedidos
) AS t3
ON
    t1.id_pedido = t3.id_pedido;

<h1>2.2 Criação das Views Analíticas</h1>

In [0]:
%sql
-- Criação da View gold.view_tempo_medio_entrega_localidade para analisar o desempenho de entrega
CREATE OR REPLACE VIEW gold.view_tempo_medio_entrega_localidade
AS
SELECT
    cidade,
    estado,
    AVG(tempo_entrega_dias) AS tempo_medio_entrega,
    AVG(tempo_entrega_estimado_dias) AS tempo_medio_estimado,
    CASE
        WHEN AVG(tempo_entrega_dias) > AVG(tempo_entrega_estimado_dias) THEN 'SIM'
        ELSE 'NÃO'
    END AS entrega_maior_que_estimado
FROM
    gold.ft_atrasos_pedidos_local_vendedor
WHERE
    entrega_no_prazo IN ('Sim', 'Não')
GROUP BY
    cidade,
    estado
ORDER BY
    tempo_medio_entrega DESC;

In [0]:
%sql
-- Criação da View gold.view_vendedor_pontualidade para ranquear vendedores por percentual de atraso
CREATE OR REPLACE VIEW gold.view_vendedor_pontualidade
AS
SELECT
    id_vendedor,
    COUNT(id_pedido) AS total_pedidos,
    SUM(CASE WHEN entrega_no_prazo = 'Não' THEN 1 ELSE 0 END) AS total_atrasados,
    (SUM(CASE WHEN entrega_no_prazo = 'Não' THEN 1 ELSE 0 END) / COUNT(id_pedido)) * 100 AS percentual_atraso
FROM
    gold.ft_atrasos_pedidos_local_vendedor
GROUP BY
    id_vendedor
ORDER BY
    percentual_atraso DESC, total_pedidos DESC;

<h1>3º Área Comercial (Análises de Vendas por Período)</h1>

# 📈 Resumo dos Objetivos das Views da Área Comercial

---

## 📅 `gold.view_vendas_por_periodo`

| Objetivo Principal | Indicadores Fornecidos | Foco Estratégico |
| :--- | :--- | :--- |
| **Análise de Tendências Temporais.** | `receita_total_brl`, `receita_total_usd` | Identificar padrões de venda (sazonalidade) ao longo de diferentes granularidades de **Tempo** (ano, trimestre, mês, dia da semana). |
| **Medição de Eficiência Média.** | `ticket_medio_brl`, `avaliacao_media` | Fornecer métricas médias para entender o valor médio das compras e a satisfação do cliente em cada período. |

---

## 🏆 `gold.view_top_produto`

| Objetivo Principal | Indicadores Fornecidos | Foco Estratégico |
| :--- | :--- | :--- |
| **Ranqueamento de Produtos/Categorias.** | `receita_brl`, `quantidade_vendida` | Avaliar e classificar **Produtos** e **Categorias** com base em seu impacto financeiro (receita) e volume. |
| **Entendimento do Cliente e Qualidade.** | `preco_medio_brl`, `avaliacao_media` | Entender o preço praticado e qual é a satisfação média associada a cada tipo de produto, direcionando decisões de *mix* e estoque. |

---

## 👗 `gold.view_vendas_produtos_esteticos`

| Objetivo Principal | Indicadores Fornecidos | Foco Estratégico |
| :--- | :--- | :--- |
| **Foco em Segmento Específico.** | `receita_total_brl`, `total_pedidos` | Criar um painel de controle exclusivo para o segmento "fashion" (estético) da empresa, permitindo uma análise granular e dedicada. |
| **Acompanhamento do Desempenho do Nicho.** | `ticket_medio_brl`, `avaliacao_media` | Monitorar como essa categoria específica se comporta ao longo do **Tempo**, isolando-a das demais vendas para análises de campanha e *marketing*. |

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dm_tempo
USING DELTA
OPTIONS ('mode' 'overwrite')
AS
WITH Max_Date_Fact AS (
    -- Busca a data máxima da coluna de compra na tabela de pedidos
    SELECT
        MAX(TO_DATE(pedido_compra_timestamp)) AS max_date
    FROM
        silver.ft_pedidos
),
dates_sequence AS (
    -- Gera a sequência de datas, partindo de uma data inicial (fixa) até a data máxima encontrada (dinâmica)
    SELECT
        EXPLODE(SEQUENCE(TO_DATE('2016-01-01'), (SELECT max_date FROM Max_Date_Fact), INTERVAL 1 DAY)) AS sk_tempo
)
SELECT
    sk_tempo,
    YEAR(sk_tempo) AS ano,
    QUARTER(sk_tempo) AS trimestre,
    MONTH(sk_tempo) AS mes,
    WEEKOFYEAR(sk_tempo) AS semana_do_ano,
    DAYOFMONTH(sk_tempo) AS dia,
    DAYOFWEEK(sk_tempo) AS dia_da_semana_num,
    CASE DAYOFWEEK(sk_tempo)
        WHEN 1 THEN 'Domingo'
        WHEN 2 THEN 'Segunda-feira'
        WHEN 3 THEN 'Terça-feira'
        WHEN 4 THEN 'Quarta-feira'
        WHEN 5 THEN 'Quinta-feira'
        WHEN 6 THEN 'Sexta-feira'
        WHEN 7 THEN 'Sábado'
    END AS dia_da_semana_nome,
    CASE MONTH(sk_tempo)
        WHEN 1 THEN 'Janeiro'
        WHEN 2 THEN 'Fevereiro'
        WHEN 3 THEN 'Março'
        WHEN 4 THEN 'Abril'
        WHEN 5 THEN 'Maio'
        WHEN 6 THEN 'Junho'
        WHEN 7 THEN 'Julho'
        WHEN 8 THEN 'Agosto'
        WHEN 9 THEN 'Setembro'
        WHEN 10 THEN 'Outubro'
        WHEN 11 THEN 'Novembro'
        WHEN 12 THEN 'Dezembro'
    END AS mes_nome,
    CASE
        WHEN DAYOFWEEK(sk_tempo) IN (1, 7) THEN 'Sim'
        ELSE 'Não'
    END AS eh_fim_de_semana
FROM
    dates_sequence;

In [0]:
%sql
CREATE OR REPLACE VIEW gold.view_vendas_por_periodo
AS
SELECT
    t2.ano,
    t2.trimestre,
    t2.mes,
    t2.mes_nome,
    t2.dia,
    t2.dia_da_semana_num,
    COUNT(DISTINCT t1.id_pedido) AS total_pedidos,
    COUNT(t1.id_item) AS total_itens,
    CAST(SUM(t1.valor_total_item_brl) AS DECIMAL(12,2)) AS receita_total_brl,
    CAST(SUM(t1.valor_total_item_usd) AS DECIMAL(12,2)) AS receita_total_usd,
    CAST(AVG(t1.valor_total_item_brl) AS DECIMAL(12,2)) AS ticket_medio_brl,
    CAST(AVG(t1.avaliacao_pedido) AS DECIMAL(3,2)) AS avaliacao_media
FROM
    gold.ft_vendas_geral AS t1
INNER JOIN
    gold.dm_tempo AS t2
ON
    t1.fk_tempo = t2.sk_tempo
GROUP BY
    t2.ano, t2.trimestre, t2.mes, t2.mes_nome, t2.dia, t2.dia_da_semana_num
ORDER BY
    t2.ano, t2.mes, t2.dia;

Consulta 1: Dia da semana com maior receita total em reais
Esta consulta utiliza o campo dia_da_semana_nome e ordena pela receita_total_brl de forma decrescente para encontrar o dia de maior faturamento.


In [0]:
%sql
SELECT
    -- Converte o número do dia da semana (1=Domingo a 7=Sábado) para o nome por extenso
    CASE t1.dia_da_semana_num
        WHEN 1 THEN 'Domingo'
        WHEN 2 THEN 'Segunda-feira'
        WHEN 3 THEN 'Terça-feira'
        WHEN 4 THEN 'Quarta-feira'
        WHEN 5 THEN 'Quinta-feira'
        WHEN 6 THEN 'Sexta-feira'
        WHEN 7 THEN 'Sábado'
    END AS dia_da_semana_nome,
    t1.receita_total_brl
FROM
    gold.view_vendas_por_periodo AS t1
ORDER BY
    t1.receita_total_brl DESC
LIMIT 1;

Consulta 2: Mês com maior ticket médio no último ano disponível
Esta consulta precisa primeiro identificar o último ano na dimensão de tempo (gold.dm_tempo) e, em seguida, filtrar e ordenar a View pelo ticket_medio_brl.

In [0]:
%sql
WITH Ultimo_Ano AS (
    --Identifica o último ano presente na Dimensão de Tempo
    SELECT
        MAX(ano) AS max_ano
    FROM
        gold.dm_tempo
)
SELECT
    t1.mes_nome,
    t1.ticket_medio_brl
FROM
    gold.view_vendas_por_periodo AS t1
INNER JOIN
    Ultimo_Ano AS t2
ON
    t1.ano = t2.max_ano
GROUP BY
    t1.mes_nome, t1.ticket_medio_brl
ORDER BY
    t1.ticket_medio_brl DESC
LIMIT 1;

## 🏆 Objetivo da View `gold.view_top_produto`

O objetivo principal desta View é fornecer uma **visão consolidada e ranqueada** da performance de vendas em nível de produto e categoria, permitindo que as áreas de **Gestão de Produtos e Comercial**  tomem decisões informadas sobre *mix* de produtos, precificação e estratégias de *marketing*

| Área de Negócio | Foco Estratégico |
| :--- | :--- |
| **Gestão de Produtos** | Entender quais produtos e categorias devem ser priorizados em estoque, desenvolvimento e onde focar a qualidade. |
| **Comercial** | Identificar os itens de maior retorno financeiro (`receita_brl`) para campanhas e avaliar o sucesso das estratégias de precificação (`preco_medio_brl`). |

---

## 🛠️ Como a View Funciona (Agregação)

A View `gold.view_top_produto` agrega os dados da Fato Geral de Vendas (`gold.ft_vendas_geral`) e os enriquece com informações da Dimensão de Produtos (assumindo a tabela `silver.dm_produtos`).

1.  **Agrupamento:** O processo central é o agrupamento de todas as transações (linhas da Fato Geral) pelas chaves **`id_produto`** e **`categoria_produto`**.
2.  **Cálculo de Receita:** Soma-se o valor total dos itens (`valor_total_item_brl` e `valor_total_item_usd`) para obter a **`receita_brl`** e **`receita_usd`** total por produto/categoria.
3.  **Cálculo de Volume:** Contam-se os itens vendidos (`quantidade_vendida`) e os pedidos distintos (`total_pedidos`) associados ao produto.
4.  **Cálculo de Médias:** Calcula-se a **`preco_medio_brl`** (preço médio do produto) e a **`avaliacao_media`** (satisfação média do cliente) para cada produto[cite: 94].

## 📊 Principais Indicadores (KPIs)

A View entrega as seguintes métricas de desempenho[cite: 94]:

| KPI | Descrição | Importância |
| :--- | :--- | :--- |
| **`receita_brl` / `receita_usd`** | Valor total faturado pelo produto/categoria. | Principal métrica para ranqueamento financeiro. |
| **`quantidade_vendida`** | Volume de itens que saiu do estoque. | Usado para gestão de inventário e logística. |
| **`avaliacao_media`** | Média das notas que os clientes deram ao produto. | Indicador-chave de qualidade e satisfação do cliente. |
| **`preco_medio_brl`** | Preço médio de venda do produto. | Essencial para análise de competitividade e rentabilidade. |

Este modelo de agregação permite que o time comercial veja rapidamente quais são os produtos "estrelas" e quais precisam de atenção (seja por baixa receita ou baixa avaliação).

In [0]:
%sql
CREATE OR REPLACE VIEW gold.view_top_produto
AS
SELECT
    t1.fk_produto AS id_produto,
    
    t2.categoria_produto,
    
    -- Métricas de Quantidade
    CAST(COUNT(t1.id_item) AS BIGINT) AS quantidade_vendida,
    CAST(COUNT(DISTINCT t1.id_pedido) AS BIGINT) AS total_pedidos,
    
    -- Métricas Financeiras
    CAST(SUM(t1.valor_total_item_brl) AS DECIMAL(12,2)) AS receita_brl,
    CAST(SUM(t1.valor_total_item_usd) AS DECIMAL(12,2)) AS receita_usd,
    
    -- Preço Médio (Média do valor_produto_brl por item)
    CAST(AVG(t1.valor_produto_brl) AS DECIMAL(12,2)) AS preco_medio_brl,
    
    -- Avaliação Média
    CAST(AVG(t1.avaliacao_pedido) AS DECIMAL(3,2)) AS avaliacao_media,
    
    -- Peso Médio
    CAST(AVG(t2.peso_produto_gramas) AS DECIMAL(8,2)) AS peso_medio_gramas
FROM
    gold.ft_vendas_geral AS t1
INNER JOIN
    silver.ft_produtos AS t2 -- 
ON
    t1.fk_produto = t2.id_produto
GROUP BY
    t1.fk_produto, t2.categoria_produto;

In [0]:
%sql
CREATE OR REPLACE VIEW gold.view_vendas_produtos_esteticos
AS
WITH Fashion_Sales AS (
    -- 1. Filtra a Fato Geral para incluir apenas categorias que iniciam com 'fashion'
    SELECT
        t1.id_pedido,
        t1.id_item,
        t1.fk_tempo,
        t2.categoria_produto,
        t1.valor_total_item_brl,
        t1.valor_total_item_usd,
        t1.avaliacao_pedido
    FROM
        gold.ft_vendas_geral AS t1
    INNER JOIN
        silver.ft_produtos AS t2
    ON
        t1.fk_produto = t2.id_produto
    WHERE
        t2.categoria_produto LIKE 'fashion%'
)
-- 2. Agrupa os resultados da CTE por Ano e Mês (via dm_tempo) e calcula os KPIs
SELECT
    t3.ano,
    t3.mes,
    t1.categoria_produto,
    
    -- Métricas de Volume
    CAST(COUNT(DISTINCT t1.id_pedido) AS BIGINT) AS total_pedidos,
    CAST(COUNT(t1.id_item) AS BIGINT) AS total_itens_vendidos,
    
    -- Métricas Financeiras
    CAST(SUM(t1.valor_total_item_brl) AS DECIMAL(12,2)) AS receita_total_brl,
    CAST(SUM(t1.valor_total_item_usd) AS DECIMAL(12,2)) AS receita_total_usd,
    
    -- Ticket Médio (por item)
    CAST(AVG(t1.valor_total_item_brl) AS DECIMAL(12,2)) AS ticket_medio_brl,
    CAST(AVG(t1.valor_total_item_usd) AS DECIMAL(12,2)) AS ticket_medio_usd,
    
    -- Avaliação Média
    CAST(AVG(t1.avaliacao_pedido) AS DECIMAL(3,2)) AS avaliacao_media
FROM
    Fashion_Sales AS t1
INNER JOIN
    gold.dm_tempo AS t3
ON
    t1.fk_tempo = t3.sk_tempo
GROUP BY
    t3.ano, t3.mes, t1.categoria_produto
ORDER BY
    t3.ano, t3.mes, receita_total_brl DESC;